# Dust Storm Event Extraction from NOAA Global Hourly / ISD Data

### Purpose
This notebook extracts and analyzes dust storm events from NOAA Global Hourly / ISD-style weather data. It identifies strict dust storm codes from station observations, clusters them into country-level events, and summarizes dust activity by day and year.

### Workflow
1. **User Inputs**: Set directories, dust storm code sets, event clustering rules, and years to process.
2. **Read Metadata**: Load station and country metadata from CSV files.
3. **Extract Observations**: Download and filter station CSVs from NOAA tar.gz archives for selected stations.
4. **Flag Dust Conditions**: Process each CSV to flag strict dust storm observations.
5. **Join Metadata**: Merge flagged observations with station metadata.
6. **Summarize Events**: Cluster observations into country-level events and summarize by day and year.
7. **Export Results**: Save summary CSVs to the output directory.


### Directory Structure
- `input/`: Station and country metadata files.
- `intermediate/`: Flagged dust storm observations by year.
- `output/`: Country-level summaries and event counts.

### Input (data sources)
- **NOAA Global Hourly / ISD**: [ISD Overview](https://www.ncei.noaa.gov/data/global-hourly/archive/csv/). The data is compressed as one `tar.gz` file per year. Each compressed file contains CSV files, one per a weather station. The CSV files consist of all coded events (see WMO codes), datetime, coordinate and many more. 
- **NOAA station metadata**: [Overview](https://www.ncei.noaa.gov/pub/data/noaa/isd-history.txt)

### Outputs
Counts of dust events per year per country:
- `dust_country_day_summary.csv`: Daily summary by country.
- `dust_country_events.csv`: Country-level dust storm events.
- `dust_country_year_dust_days.csv`: Yearly dust days by country.
- `dust_country_year_events.csv`: Yearly event counts by country.
- `dust_strict_observations.csv`: All flagged strict dust storm observations.

### Preset parameters
- **STRICT_DUST_CODES**: `{9, 30, 31, 32, 33, 34, 35, 98}`
- **BROAD_DUST_CODES**: Includes additional dust-related codes.
- See more in [WMO Present Weather Codes](https://www.nodc.noaa.gov/archive/arc0021/0002199/1.1/data/0-data/HTML/WMO-CODE/WMO4677.HTM).
- **event_gap_hours**: Hours between observations to cluster as a single event (default: 24).
- **min_unique_stations**: Minimum stations required to confirm an event.

### Notes
- The extraction uses streaming to handle large archives efficiently in case of large number of years of data.
- Adjust code sets and event rules for your research needs.
- Example content for `country-region.csv`:
    ```
    Region,Country,ISO3,ISO2
    West Africa,Nigeria,NGA,NG
    West Africa,Ghana,GHA,GH
    ```

### Usage
1. Get station metadata and country-region CSVs (see above) to the `input/` directory.
2. Run the notebook: `dust storm extraction.ipynb`, cell by cell
3. Adjust parameters as needed for your analysis.
4. Results will be saved in the `output/` directory.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import requests
import tarfile
import os
from io import BytesIO
import time
import glob
import re

## Step 1: User input

In [ ]:
input_directory = "./input"

stations_file = f"{input_directory}/isd-history.txt"        # your station metadata
country_file = f"{input_directory}/country-region.csv"        # your country metadata

# Strict dust storm codes from WMO present weather table:
# 09 = duststorm/sandstorm in sight or at station during preceding hour
# 30-35 = slight/moderate/severe duststorm or sandstorm
# 98 = thunderstorm combined with duststorm/sandstorm
STRICT_DUST_CODES = {9, 30, 31, 32, 33, 34, 35, 98}

# Broader dust-related weather conditions:
# 06 = widespread dust in suspension
# 07 = dust/sand raised by wind, no storm seen
# 08 = dust/sand whirls, no storm seen
BROAD_DUST_CODES = STRICT_DUST_CODES | {6, 7, 8}

# Event clustering rule
event_gap_hours = 24

# Optional filter: require at least this many unique stations in a country-event
min_unique_stations = 1

# Years of data to process (adjust as needed, or set to None for all years in obs_file)
years_to_process = [y for y in range(1971, 2013)]  # originally 1971, 2013

## Step 2: Read data

In [ ]:
# Country ISO codes
countries = pd.read_csv(country_file)
countries.columns = countries.columns.str.lower()
ctry_interested = list(countries["iso2"])

# Stations
column_names = ["USAF", "WBAN", "STATION NAME", "CTRY", "ST", "CALL", "LAT", "LON", "ELEV(M)", "BEGIN", "END"]
stations = pd.read_fwf(
    stations_file,
    skiprows=21,
    names=column_names,
    infer_nrows=10000, # to include data that has all columns filled
    dtype=str,
    on_bad_lines="warn",
)

stations["station_id"] = stations["USAF"].str.strip() + stations["WBAN"].str.strip()
stations = stations.rename(columns={
    "station_id": "station_id",
    "CTRY": "country"
})

# Get station IDs per countries of interest
stations_filtered = stations[stations["country"].isin(ctry_interested)].reset_index(drop=True)
station_ids = stations_filtered['station_id'].astype(str).tolist()

## Step 3: Extract NOAA station observation

In [ ]:
''' 
Get data csvs only selected stations from online NOAA Global Hourly / ISD-style tar.gz files
and save csvs locally
The Extraction uses streaming method (for large archives without enough disk space)
'''

intermediate_dir = f'./intermediate'

for year in years_to_process:

    print(f"Extracting year {year}...")
    output_dir = f'{intermediate_dir}/obs/{year}'  # Directory to extract filtered files
    os.makedirs(output_dir, exist_ok=True)

    # Example tar.gz URL (update as needed)
    tar_url = f'https://www.ncei.noaa.gov/data/global-hourly/archive/csv/{year}.tar.gz'
    output_dir = f'{intermediate_dir}/obs/{year}'  # Directory to extract filtered files

    # Stream the tar.gz file
    with requests.get(tar_url, stream=True) as r:
        r.raise_for_status()
        # Use a streaming file object
        fileobj = BytesIO()
        for chunk in r.iter_content(chunk_size=1024*1024):  # 1MB chunks
            fileobj.write(chunk)
        fileobj.seek(0)
        with tarfile.open(fileobj=fileobj, mode='r:gz') as tar:
            for member in tar.getmembers():
                if member.name.endswith('.csv'):
                    station_name = os.path.splitext(os.path.basename(member.name))[0]
                    if station_name in station_ids:
                        tar.extract(member, path=output_dir)
                        print(f"Extracted {member.name}")

    time.sleep(1)  # Optional: pause between years to avoid overwhelming the server
    print(f"Finished extracting year {year}.")

print("Selective extraction complete.")

## Step 4: Extract dust conditions

In [ ]:
'''
Process each CSV individually to flag dust conditions and save results incrementally
'''

obs_dir = os.path.join(intermediate_dir, "obs")

def extract_codes(row):
    vals = []
    for c in present_weather_cols:
        v = row[c]
        if pd.notna(v):
            vals.append(int(v))
    return sorted(set(vals))

def has_any_code(codes, code_set):
    return any(code in code_set for code in codes)

def parse_code(x):
    if pd.isna(x):
        return np.nan
    m = re.search(r"(\d{1,2})", str(x))
    return int(m.group(1)) if m else np.nan

def concat_dfs(df1, df2):
    all_cols = df1.columns.union(df2.columns)
    df1 = df1.reindex(columns=all_cols)
    df2 = df2.reindex(columns=all_cols)
    return pd.concat([df1, df2], ignore_index=True)

first_write = True
dust_obs = pd.DataFrame()  # To accumulate flagged observations
for year in years_to_process[:1]:
    all_csvs = glob.glob(os.path.join(obs_dir, f"{year}/*.csv"))
    for f in all_csvs:
        print(f"Processing {f}...")
        try:
            df = pd.read_csv(f, engine='python')
        except Exception as e:
            print(f"Error reading {f}: {e}")
            continue
        df['STATION'] = df['STATION'].astype(str)  # Ensure station ID is read as string
        df["datetime"] = pd.to_datetime(df["DATE"], errors="coerce", utc=True)

        # 3. PREPARE PRESENT WEATHER CODES
        present_weather_cols = [c for c in df.columns if re.fullmatch(r"(AW|MW|AU)\d+", c)]
        if not present_weather_cols:
            continue  # Skip files with no present weather columns
        for c in present_weather_cols:
            df[c] = df[c].apply(parse_code)
        df["wx_codes"] = df.apply(extract_codes, axis=1)
        # 4. FLAG DUST CONDITIONS
        df["has_strict_dust"] = df["wx_codes"].apply(lambda x: has_any_code(x, STRICT_DUST_CODES))
        flagged = df.loc[df["has_strict_dust"]].copy()
        if not flagged.empty:
            dust_obs = concat_dfs(dust_obs, flagged)
            all_cols = dust_obs.columns
            flagged = flagged.reindex(columns=all_cols)
    
    # Output file for flagged dust observations
    output_file = f"dust_strict_observations_{year}.csv"
    dust_obs.to_csv(
        f"{output_dir}/{output_file}", 
        mode='w',
        sep=";",
        index=False,
        columns=all_cols
    )

    print(f"Flagged dust observations written to {output_file}")

## 4.1 [optional] Extract NOAA station observations (read back flagged csvs)

In [ ]:
'''
In case you want to read back the flagged observations for analysis
Otherwise you can skip this part and directly analyze the `dust_obs` DataFrame in memory
'''

output_file = "dust_strict_observations_*.csv"

dust_obs_files = glob.glob(os.path.join(intermediate_dir, f"{output_file}"))

# Read each file into a DataFrame
dfs = [pd.read_csv(f, sep=";") for f in dust_obs_files]

# Concatenate all DataFrames
dust_obs = pd.concat(dfs, ignore_index=True)

# dust_obs = pd.read_csv(f"{directory}/{output_file}", sep=";")
dust_obs['STATION'] = dust_obs['STATION'].astype(str)  # Ensure station ID is string
dust_obs['datetime'] = pd.to_datetime(dust_obs['DATE'], errors='coerce', utc=True)

## Step 5: Process and count events per country

In [ ]:
# ----------------------------
# JOIN STATION METADATA
# ----------------------------

dust_obs = dust_obs.merge(
    stations[["station_id", "country"]].drop_duplicates(),
    left_on="STATION",
    right_on="station_id",
    how="left"
)

# Optional: drop rows without country
dust_obs = dust_obs.loc[dust_obs["country"].notna()].copy()

# ----------------------------
# CREATE COUNTRY-DAY SUMMARY
# ----------------------------

dust_obs["date"] = dust_obs["datetime"].dt.date

country_day = (
    dust_obs.groupby(["country", "date"], as_index=False)
    .agg(
        n_obs=("station_id", "size"),
        n_stations=("station_id", pd.Series.nunique)
    )
    .sort_values(["country", "date"])
)

# ----------------------------
# RECONSTRUCT COUNTRY-LEVEL EVENTS
# ----------------------------

# Event rule:
# Sort strict dust-storm observations by country and time.
# A new event starts if the gap from the previous dust observation in the same country
# is greater than event_gap_hours.

dust_obs = dust_obs.sort_values(["country", "datetime"]).copy()

dust_obs["prev_datetime"] = dust_obs.groupby("country")["datetime"].shift(1)
dust_obs["gap_hours"] = (
    (dust_obs["datetime"] - dust_obs["prev_datetime"]).dt.total_seconds() / 3600
)

dust_obs["new_event"] = (
    dust_obs["prev_datetime"].isna() |
    (dust_obs["gap_hours"] > event_gap_hours)
)

dust_obs["event_number"] = dust_obs.groupby("country")["new_event"].cumsum()

events = (
    dust_obs.groupby(["country", "event_number"], as_index=False)
    .agg(
        start_time=("datetime", "min"),
        end_time=("datetime", "max"),
        n_obs=("station_id", "size"),
        n_stations=("station_id", pd.Series.nunique),
        stations=("station_id", lambda x: ",".join(sorted(set(map(str, x))))),
        wx_codes_seen=("wx_codes", lambda s: sorted(set(code for codes in s for code in codes)))
    )
)

events["duration_hours"] = (
    (events["end_time"] - events["start_time"]).dt.total_seconds() / 3600
)

# Optional quality filter: require minimum number of unique stations
events = events.loc[events["n_stations"] >= min_unique_stations].copy()

# ----------------------------
# COUNTRY-YEAR COUNTS
# ----------------------------

events["year"] = events["start_time"].dt.year

country_year_events = (
    events.groupby(["country", "year"], as_index=False)
    .agg(
        n_events=("event_number", "count"),
        total_station_reports=("n_obs", "sum"),
        mean_stations_per_event=("n_stations", "mean")
    )
)

country_year_dust_days = (
    country_day.assign(year=pd.to_datetime(country_day["date"]).dt.year)
    .groupby(["country", "year"], as_index=False)
    .agg(
        dust_days=("date", "nunique"),
        total_station_days=("n_stations", "sum")
    )
)


## Step 6: Export results in CSVs


In [ ]:

country_day.to_csv("dust_country_day_summary.csv", index=False)
events.to_csv("dust_country_events.csv", index=False)
country_year_events.to_csv("dust_country_year_events.csv", index=False)
country_year_dust_days.to_csv("dust_country_year_dust_days.csv", index=False)

print("Done.")
print(f"Strict dust observations: {len(dust_obs):,}")
print(f"Country-level events: {len(events):,}")
print(country_year_events.head(20))